# E2.4 · Sector overlays

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.3 · Voluntary frameworks as your spine](https://spbreed.github.io/cyber-commons/lessons/E2.3.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Map one agent to existing model-risk obligations.

**Why a security engineer needs it.** An agent is already a "model" under model-risk rules you already comply with. The control it builds is: find the regime you're already in before inventing a new one.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Sector overlays add requirements rather than replacing them. Reconciling them against the common spine is what stops a financial-services obligation and a healthcare obligation becoming two separate control estates.

> **At CyberTravels.** A travel company touches payment rules, privacy law and, through corporate health bookings, health obligations. Overlays add; they do not replace.

## 2 · The framework

```
   overlays ADD, they do not replace

   +---------------------------------------+
   |          the common spine             |
   +---------------------------------------+
      + finance: model risk, records
      + health: safety, clinical validation
      + critical infra: resilience, reporting

   reconcile at the spine, or you end up with three control estates
```

Sector overlays usually bite first, and the reason is structural: they already
applied before anyone deployed an agent, they already have a supervisor who
knows your organisation, and several of their clauses cover autonomous action
without ever using the word AI.

Four clause types that catch agents without naming them:

- **ICT third-party risk** (DORA) — your model provider is an ICT third party.
- **Exit strategy** (DORA) — can you stop using this provider? Most AI contracts
  have no answer.
- **Scope containment** (PCI DSS) — an agent with access to the cardholder data
  environment expands that environment.
- **Minimum necessary** (HIPAA) — the agent's context window is a disclosure.

Citing an existing clause is also far more effective internally than proposing a
new AI policy: it needs no new governance, and somebody already owns it.

## 3 · The procedure, as a skill

Seven clauses already bind the claims-triage agent and none of them mentions AI. The skill searches on function rather than on technology, and assesses whether the hosted frontier API could be exited at all.

In [ ]:
# skills/regulatory/sector-overlay-assessment/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: sector-overlay-assessment
description: >-
  Find the pre-existing sector clauses that already apply to an AI system
  without mentioning AI, and assess provider exit against them. Use when a team
  believes no regulation applies yet because the AI rules are not in force.
allowed-tools: Read, Grep, Glob
---

# The clauses that already apply and never say "AI"

Sector regulation was written before this and applies anyway. Outsourcing,
material change, operational resilience, exit strategy, data handling — none of
them mention AI, all of them bind an agent that processes claims or moves money.
The finding is usually that a system nobody thought was regulated attracts seven
clauses today.

## When to use this

Any AI system in a regulated sector, especially one whose team believes the
obligations arrive later.

## Procedure

**1 — Describe the system functionally.** What it processes, what it decides,
whose money or health or data it touches. Sector clauses trigger on function,
not on technology.

**2 — Search existing obligations for the functional triggers.** Outsourcing and
third-party dependency, material change notification, resilience and continuity,
record keeping, exit. Do not search for "AI" — that is why nobody found them.

**3 — Record each clause with what it requires,** in operational terms. "Notify
the regulator of material change to an outsourced arrangement" becomes a
question about whether a hosted model version change is a material change.

**4 — Assess provider exit specifically.** Could you move off this provider,
how long would it take, and is the alternative viable? A hosted frontier API
with no equivalent substitute usually fails an exit test that a database would
pass.

**5 — Make the case for each clause explicitly.** Clause, why it applies, what
it requires here. That framing survives a challenge from a team that would
rather it did not apply.

## Output contract

```json
{
  "system": {"function": "str", "decides": "str", "data": ["str"]},
  "clauses": [{"framework": "str", "clause": "str", "mentions_ai": false,
               "applies_because": "str", "requires": "str"}],
  "exit": [{"provider": "str", "substitutable": false, "days_to_exit": 0, "verdict": "str"}],
  "cases": [{"clause": "str", "argument": "str"}]
}
```

## Failure modes

- **Searching for AI.** The binding clauses do not use the word.
- **Waiting for the horizontal regime.** These apply now.
- **An exit assessment that assumes substitutability.** Test it.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/regulatory/sector-overlay-assessment/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/regulatory/sector-overlay-assessment/scripts/sector_overlay_assessment.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Find the pre-existing sector clauses that already apply to an AI system without mentioning AI, and assess provider exit.

This is the executable half of the `sector-overlay-assessment` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

OVERLAYS = {
 "DORA (financial)": [
   ("ICT third-party risk", "your model provider is an ICT third party",
    lambda s: s["uses_external_model"]),
   ("exit strategy", "can you stop using this provider and keep operating?",
    lambda s: s["uses_external_model"]),
   ("resilience testing", "your stop mechanism is in scope for testing",
    lambda s: s["autonomy"] in ("L2.5", "L3")),
   ("incident reporting", "clocks measured in hours",
    lambda s: True)],
 "HIPAA (health)": [
   ("minimum necessary", "the context window is a disclosure",
    lambda s: "health" in s["data"]),
   ("audit controls", "the ACTING identity must be recorded",
    lambda s: True)],
 "PCI DSS (cards)": [
   ("scope containment", "an agent with CDE access expands the CDE",
    lambda s: "cardholder" in s["data"]),
   ("access control", "non-human identities need the same rigour",
    lambda s: True)],
}
SYSTEM = {"name": "claims-triage-agent", "uses_external_model": True,
          "autonomy": "L2.5", "data": ("customer", "health")}

print(f"{SYSTEM['name']}: autonomy {SYSTEM['autonomy']}, data {list(SYSTEM['data'])}, "
      f"external model {SYSTEM['uses_external_model']}\n")
hits = []
for fw, clauses in OVERLAYS.items():
    applicable = [(c, why) for c, why, test in clauses if test(SYSTEM)]
    if not applicable: continue
    print(f"{fw}")
    for c, why in applicable:
        hits.append((fw, c)); print(f"   {c:24s}{why}")
    print()
print(f"{len(hits)} pre-existing clauses apply. None of them mentions AI.")

PROVIDERS = {
 "hosted frontier API": {"can_pin_version": False, "can_export_weights": False,
                         "equivalent_alternative": True, "switching_days": 45},
 "hosted open-weight API": {"can_pin_version": True, "can_export_weights": False,
                            "equivalent_alternative": True, "switching_days": 14},
 "self-hosted open weights": {"can_pin_version": True, "can_export_weights": True,
                              "equivalent_alternative": True, "switching_days": 2},
}
def exit_assessment(p):
    problems = []
    if not p["can_pin_version"]:
        problems.append("cannot pin a version — behaviour changes without notice")
    if not p["can_export_weights"]:
        problems.append("cannot retain the artefact — no continuity if withdrawn")
    if p["switching_days"] > 30:
        problems.append(f"{p['switching_days']}d to switch — outside most RTOs")
    return (not problems), problems

print(f"{'provider':28s}{'exit strategy':>15}")
print("-" * 48)
for name, p in PROVIDERS.items():
    ok, problems = exit_assessment(p)
    print(f"{name:28s}{'defensible' if ok else 'NOT DEFENSIBLE':>15}")
    for x in problems: print(f"      ⚠ {x}")
print("\nDORA Art.11 asks this directly. It is the clause most AI procurement")
print("cannot answer, and it was written years before anyone deployed an agent.")

def make_the_case(clause, framework, agent_fact, existing_owner):
    return (f"'{clause}' ({framework}) already applies to us and is owned by "
            f"{existing_owner}.\n"
            f"   The agent fact that engages it: {agent_fact}\n"
            f"   Ask: extend the existing control, not create an AI policy.")

CASES = [
 ("ICT third-party risk", "DORA", "the model provider is an ICT third party",
  "third-party risk management"),
 ("audit controls", "HIPAA", "the acting identity is not currently recorded",
  "the security team"),
 ("access control", "PCI DSS", "non-human identities have no recertification",
  "identity and access management"),
]
for c, fw, fact, owner in CASES:
    print(make_the_case(c, fw, fact, owner)); print()

print("Compare with the alternative ask:")
print("   'We need a new AI governance policy and a new committee.'")
print("   → new owner, new process, new budget line, six months.")
print("versus")
print("   'Extend third-party risk to cover model providers.'")
print("   → existing owner, existing process, next review cycle.")
assert len(CASES) == 3

## What you just proved

Seven pre-existing clauses apply to the claims-triage agent across DORA, HIPAA and PCI DSS, none of which mentions AI. The exit-strategy assessment marks the hosted frontier API as not defensible on all three counts, the hosted open-weight API on one, and self-hosted weights as defensible. Three cases show how to route the requirement to an existing owner rather than a new policy.

## Your turn

Find the clause in your own sector overlay that already covers autonomous action without naming AI. Citing it is faster, cheaper and more persuasive than any new AI policy you could write.

---

**Next → [E2.5 · Privacy and data protection](https://spbreed.github.io/cyber-commons/lessons/E2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*